In [1]:
from datetime import datetime, timedelta, date
import requests
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from rapidfuzz import fuzz
from pandas import json_normalize
import retrying
from tqdm.notebook import tqdm
import pdfkit
import aiohttp
import asyncio
import xmltodict
import warnings
from dash import Dash, html, Input, Output, dash_table, dcc
from jupyter_dash import JupyterDash
import dash_bootstrap_components as dbc
import plotly.express as px
import plotly.io as pio
import json
warnings.filterwarnings("ignore")

In [2]:
user = 'xxxxxxxx'
password = 'xxxxxxxx'

In [3]:
tomorrow = (datetime.now() + timedelta(days=1)).strftime('%Y%m%d')
print(tomorrow)

20261021


In [4]:
column_mapping = {'@Id': 'Id', '@Status': 'Status', '@BookingCode': 'BookingCode', '@BookingDate': 'BookingDate',
                  '@LastModifiedDate': 'LastModifiedDate','@AgencyRef':'AgencyRef','@TimeZone':'TimeZone'}

def get_booking_codes(tomorrow):
    url="https://xxxxxxxxx/wsExportacion/wsBookings.asmx?op=getBookingList"
    headers = {'content-type': 'text/xml','SOAPAction':'http://juniper.es/getBookingList',"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:66.0) Gecko/20100101 Firefox/66.0",
        "Accept-Encoding": "*",
        "Connection": "keep-alive"}
    body = f"""<?xml version="1.0" encoding="utf-8"?>
        <soap:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
          <soap:Body>
            <getBookingList xmlns="http://juniper.es/">
              <user>{user}</user>
              <password>{password}</password>
              <BeginTravelDateFrom>{tomorrow}</BeginTravelDateFrom>
              <NonRefundable>0</NonRefundable>
              <Status>OK,Con</Status>
            </getBookingList>
          </soap:Body>
        </soap:Envelope>"""
    while True:
        try:
            response = requests.post(url,data=body,headers=headers)
            break
        except:
            pass
    data = xmltodict.parse(response.text)
    listBookingsDF = pd.DataFrame(data['soap:Envelope']['soap:Body']['getBookingListResponse']['getBookingListResult']['wsResult']['BookingList']['Booking'])
    return listBookingsDF
listBookingsDF = get_booking_codes(tomorrow)

In [5]:
listBookingsDF = listBookingsDF.rename(columns=column_mapping)
bookingCodes = list(set(listBookingsDF['BookingCode'].to_list()))
len(bookingCodes)

3321

In [6]:
df = pd.DataFrame(columns=['BookingCode','Product Group','Agency','Line Id','JPCode','HotelCode','HotelName','Line Status','Internal Remarks','Booking Remarks',
                           'Travel Date', 'End Date','Nationality Id','Nationality', 'No. Paxes', 'Supplier Name',
                           'Days To Cancel','Board Type','Room Name','No_Rooms','Non Refund','Net Cost(USD)','Sell Rate(USD)','Room_data','Ages','Room_Names'])

@retrying.retry(stop_max_attempt_number=3, wait_exponential_multiplier=1000, wait_exponential_max=10000)
async def GetBookingData(session, code, pbar):
    url = "https://xxxxxxxxxx/wsExportacion/wsBookings.asmx"
    headers = {
        'content-type': 'text/xml',
        'SOAPAction': "http://juniper.es/getBookings",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:66.0) Gecko/20100101 Firefox/66.0",
        "Accept-Encoding": "*",
        "Connection": "keep-alive"
    }
    body = f"""<?xml version="1.0" encoding="utf-8"?>
        <soap:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
          <soap:Body>
            <getBookings xmlns="http://juniper.es/">
              <user>{user}</user>
              <password>{password}</password>
              <BookingCode>{code}</BookingCode>
            </getBookings>
          </soap:Body>
        </soap:Envelope>"""
    try:
        async with session.post(url, data=body, headers=headers) as response:
            responseDict = xmltodict.parse(await response.text())
            AllBookings = responseDict['soap:Envelope']['soap:Body']['getBookingsResponse']['getBookingsResult']['wsResult']['Bookings']['Booking']
            if AllBookings['Customer']['Name'] != 'Global International':
                if isinstance(AllBookings['Lines']['Line'], list):
                    for j in range(len(AllBookings['Lines']['Line'])):
                        ages = []
                        roomnames = []
                        if isinstance(AllBookings['Lines']['Line'][j]['Paxes']['Pax'], list):
                            for lp in AllBookings['Lines']['Line'][j]['Paxes']['Pax']:
                                ages.append(lp['Age'])
                        else:
                            ages.append(AllBookings['Lines']['Line'][j]['Paxes']['Pax']['Age'])
                        if isinstance(AllBookings['Lines']['Line'][j]['roomlist']['room'], list):
                            rooms = len(AllBookings['Lines']['Line'][j]['roomlist']['room'])
                            jpcode = AllBookings['Lines']['Line'][j]['roomlist']['room'][0]['JPCode']
                            hotelcode = AllBookings['Lines']['Line'][j]['roomlist']['room'][0]['hotelcode']
                            board = AllBookings['Lines']['Line'][j]['roomlist']['room'][0]['boardtype']['#text']
                            room = AllBookings['Lines']['Line'][j]['roomlist']['room'][0]['typeroomname']
                            room_data = {}

                            for r in range(1,rooms+1):
                                roomnames.append(AllBookings['Lines']['Line'][j]['roomlist']['room'][r-1]['typeroomname'])
                                adult=0
                                child=0
                                baby=0
                                if isinstance(AllBookings['Lines']['Line'][j]['roomlist']['room'][r-1]['paxes']['pax'], list):
                                    for p in range(len(AllBookings['Lines']['Line'][j]['roomlist']['room'][r-1]['paxes']['pax'])):
                                        pax_type = AllBookings['Lines']['Line'][j]['roomlist']['room'][r-1]['paxes']['pax'][p]['typepax']
                                        if pax_type == 'Adult':
                                            adult+=1
                                        if pax_type == 'Child':
                                            child+=1
                                        if pax_type == 'Baby':
                                            child+=1
                                else:
                                    pax_type = AllBookings['Lines']['Line'][j]['roomlist']['room'][r-1]['paxes']['pax']['typepax']
                                    if pax_type == 'Adult':
                                        adult+=1
                                    if pax_type == 'Child':
                                        child+=1
                                    if pax_type == 'Baby':
                                        child+=1
                                
                                room_data[r] = [adult, child]
                        else:
                            rooms = 1
                            jpcode=AllBookings['Lines']['Line'][j]['roomlist']['room']['JPCode']
                            hotelcode=AllBookings['Lines']['Line'][j]['roomlist']['room']['hotelcode']
                            board = AllBookings['Lines']['Line'][j]['roomlist']['room']['boardtype']['#text']
                            room = AllBookings['Lines']['Line'][j]['roomlist']['room']['typeroomname']
                            roomnames.append(AllBookings['Lines']['Line'][j]['roomlist']['room']['typeroomname'])
                            room_data = {}
                            adult=0
                            child=0
                            baby=0
                            if isinstance(AllBookings['Lines']['Line'][j]['roomlist']['room']['paxes']['pax'], list):
                                for p in range(len(AllBookings['Lines']['Line'][j]['roomlist']['room']['paxes']['pax'])):
                                    pax_type = AllBookings['Lines']['Line'][j]['roomlist']['room']['paxes']['pax'][p]['typepax']
                                    if pax_type == 'Adult':
                                        adult+=1
                                    if pax_type == 'Child':
                                        child+=1
                                    if pax_type == 'Baby':
                                        child+=1
                            else:
                                pax_type = AllBookings['Lines']['Line'][j]['roomlist']['room']['paxes']['pax']['typepax']
                                if pax_type == 'Adult':
                                    adult+=1
                                if pax_type == 'Child':
                                    child+=1
                                if pax_type == 'Baby':
                                    child+=1
                                
                            room_data[1] = [adult, child]
                        room_data = json.dumps(room_data)
                        row = [
                            AllBookings['@BookingCode'],
                            AllBookings['Lines']['Line'][j]['ProductGroupName'],
                            AllBookings['Customer']['Name'],
                            AllBookings['Lines']['Line'][j]['@IdBookLine'],
                            jpcode,
                            hotelcode,
                            AllBookings['Lines']['Line'][j]['ServiceName'],
                            AllBookings['Lines']['Line'][j]['@LineCancelled'],
                            AllBookings['InRemarks'],
                            AllBookings['Remarks'],
                            AllBookings['Lines']['Line'][j]['BeginTravelDate'][:AllBookings['Lines']['Line'][j]['BeginTravelDate'].index('T')],
                            AllBookings['Lines']['Line'][j]['EndTravelDate'][:AllBookings['Lines']['Line'][j]['EndTravelDate'].index('T')],
                            AllBookings['Holder']['Nacionalidad']['@ISO2'],
                            AllBookings['Holder']['Nacionalidad']['#text'],
                            AllBookings['Lines']['Line'][j]['PaxNumber'],
                            AllBookings['Lines']['Line'][j]['Supplier']['SupplierName'],
                            (datetime.strptime(AllBookings['Lines']['Line'][j]['@LineCancellationChargesDate'], '%Y-%m-%dT%H:%M:%S')-datetime.now()).days,
                            board,
                            room,
                            rooms,
                            AllBookings['Lines']['Line'][j]['@NonRefundable'],
                            float(AllBookings['Lines']['Line'][j]['CostChangeFactor'])*float(AllBookings['Lines']['Line'][j]['NetCostLine']),
                            AllBookings['Lines']['Line'][j]['SellingPrice'],
                            room_data,
                            ages,
                            roomnames
                        ]
                        df.loc[len(df)] = row
                else:
                    ages = []
                    roomnames = []
                    if isinstance(AllBookings['Lines']['Line']['Paxes']['Pax'], list):  
                        for lp in (AllBookings['Lines']['Line']['Paxes']['Pax']):
                            ages.append(lp['Age'])
                    else:
                        ages.append(AllBookings['Lines']['Line']['Paxes']['Pax']['Age'])
                    if isinstance(AllBookings['Lines']['Line']['roomlist']['room'], list):
                        rooms = len(AllBookings['Lines']['Line']['roomlist']['room'])
                        jpcode = AllBookings['Lines']['Line']['roomlist']['room'][0]['JPCode']
                        hotelcode = AllBookings['Lines']['Line']['roomlist']['room'][0]['hotelcode']
                        board = AllBookings['Lines']['Line']['roomlist']['room'][0]['boardtype']['#text']
                        room = AllBookings['Lines']['Line']['roomlist']['room'][0]['typeroomname']
                        room_data = {}
                        for r in range(1,rooms+1):
                            roomnames.append(AllBookings['Lines']['Line']['roomlist']['room'][r-1]['typeroomname'])
                            adult=0
                            child=0
                            baby=0
                            if isinstance(AllBookings['Lines']['Line']['roomlist']['room'][r-1]['paxes']['pax'], list):
                                for p in range(len(AllBookings['Lines']['Line']['roomlist']['room'][r-1]['paxes']['pax'])):
                                    pax_type = AllBookings['Lines']['Line']['roomlist']['room'][r-1]['paxes']['pax'][p]['typepax']
                                    if pax_type == 'Adult':
                                        adult+=1
                                    if pax_type == 'Child':
                                        child+=1
                                    if pax_type == 'Baby':
                                        child+=1
                            else:
                                pax_type = AllBookings['Lines']['Line']['roomlist']['room'][r-1]['paxes']['pax']['typepax']
                                if pax_type == 'Adult':
                                    adult+=1
                                if pax_type == 'Child':
                                    child+=1
                                if pax_type == 'Baby':
                                    child+=1
    
                            room_data[r] = [adult, child]
                    else:
                        rooms = 1
                        jpcode = AllBookings['Lines']['Line']['roomlist']['room']['JPCode']
                        hotelcode = AllBookings['Lines']['Line']['roomlist']['room']['hotelcode']
                        board = AllBookings['Lines']['Line']['roomlist']['room']['boardtype']['#text']
                        room = AllBookings['Lines']['Line']['roomlist']['room']['typeroomname']
                        roomnames.append(AllBookings['Lines']['Line']['roomlist']['room']['typeroomname'])
                        room_data = {}
                        adult=0
                        child=0
                        baby=0
                        if isinstance(AllBookings['Lines']['Line']['roomlist']['room']['paxes']['pax'], list):
                            for p in range(len(AllBookings['Lines']['Line']['roomlist']['room']['paxes']['pax'])):
                                pax_type = AllBookings['Lines']['Line']['roomlist']['room']['paxes']['pax'][p]['typepax']
                                if pax_type == 'Adult':
                                    adult+=1
                                if pax_type == 'Child':
                                    child+=1
                                if pax_type == 'Baby':
                                    child+=1
                        else:
                            pax_type = AllBookings['Lines']['Line']['roomlist']['room']['paxes']['pax']['typepax']
                            if pax_type == 'Adult':
                                adult+=1
                            if pax_type == 'Child':
                                child+=1
                            if pax_type == 'Baby':
                                child+=1
                            
                        room_data[1] = [adult, child]
                    room_data = json.dumps(room_data)
                    row = [
                    AllBookings['@BookingCode'],
                    AllBookings['Lines']['Line']['ProductGroupName'],
                    AllBookings['Customer']['Name'],
                    AllBookings['Lines']['Line']['@IdBookLine'],
                    jpcode,
                    hotelcode,
                    AllBookings['Lines']['Line']['ServiceName'],
                    AllBookings['Lines']['Line']['@LineCancelled'],
                    AllBookings['InRemarks'],
                    AllBookings['Remarks'],
                    AllBookings['Lines']['Line']['BeginTravelDate'][:AllBookings['Lines']['Line']['BeginTravelDate'].index('T')],
                    AllBookings['Lines']['Line']['EndTravelDate'][:AllBookings['Lines']['Line']['EndTravelDate'].index('T')],
                    AllBookings['Holder']['Nacionalidad']['@ISO2'],
                    AllBookings['Holder']['Nacionalidad']['#text'],
                    AllBookings['Lines']['Line']['PaxNumber'],
                    AllBookings['Lines']['Line']['Supplier']['SupplierName'],
                    (datetime.strptime(AllBookings['Lines']['Line']['@LineCancellationChargesDate'], '%Y-%m-%dT%H:%M:%S')-datetime.now()).days,
                    board,
                    room,
                    rooms,
                    AllBookings['Lines']['Line']['@NonRefundable'],
                    float(AllBookings['Lines']['Line']['CostChangeFactor'])*float(AllBookings['Lines']['Line']['NetCostLine']),
                    AllBookings['Lines']['Line']['SellingPrice'],
                    room_data,
                    ages,
                    roomnames
                    ]
                    df.loc[len(df)] = row
    except Exception as e:
        pass
    pbar.update(1)


async def main():
#     bookingCodes = ['BJSFKK']#list(set(listBookingsDF['BookingCode'].to_list()))
    for i in tqdm(range(0,len(bookingCodes),1000)):
        new_codes = bookingCodes[i:i+1000]
        async with aiohttp.ClientSession() as session:
            tasks = []
            with tqdm(total=len(new_codes), desc="Processing") as pbar:
                for code in new_codes:
                    tasks.append(asyncio.ensure_future(GetBookingData(session, code, pbar)))
                await asyncio.gather(*tasks)

await main()

  0%|          | 0/4 [00:00<?, ?it/s]

Processing:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing:   0%|          | 0/321 [00:00<?, ?it/s]

In [7]:
df.shape

(3145, 26)

In [8]:
# cleaned_df = df[(df['Non Refund']=='False') & (~df['Internal Remarks'].str.contains('convert|related', na=False,case=False)) \
#   & (df['Line Status']=='False')]

In [9]:
third_party_suppliers=['H2H - Expedia','Lots Of Hotels','Hotel Beds','H2H - Destination of the world','Derbysoft Radisson (H2H)'
,'Hilton External Supplier','HCC - DerbySoft Best Western','HCC - DerbySoft Rotana','Rotana (DerbySoft)',
'HCD-Sabre Shangri La via','H2H - TDSTravel (Non-KSA)','H2H - TDSTravel (KSA)','HiltonV2']

In [10]:
cleaned_df = df[(df['Non Refund']=='False') & (~df['Internal Remarks'].str.contains('convert|related', na=False,case=False)) \
  & (df['Line Status']=='False') & (~df['Agency'].str.contains('Global International|How To Work|Contracting Test',na=False,case=False)) & ((df['Supplier Name'].isin(third_party_suppliers))|(df['Supplier Name'].str.contains("~"))|(df['Supplier Name'].str.contains("Dynamic")))]

In [11]:
#cleaned_df = cleaned_df[cleaned_df['No_Rooms']==1]

In [12]:
cleaned_df = cleaned_df.reset_index()

In [13]:
cleaned_df['No_Rooms'].unique()

array([1, 3, 2, 5, 4], dtype=int64)

In [14]:
from HotelsAvailabiltyApi import *
from HotelsAvailabiltyApi import main

In [15]:
availability_df = pd.DataFrame()
for row,v in tqdm(cleaned_df.iterrows(),total=len(cleaned_df)):
    all_ages = cleaned_df.at[row,'Ages']
    rooms = cleaned_df.at[row,'No_Rooms']
    x=""
    total_paxes = 0
    for r in range(1,rooms+1):
        adults_no = json.loads(cleaned_df.at[row,'Room_data'])[str(r)][0]
        child_no = json.loads(cleaned_df.at[row,'Room_data'])[str(r)][1]
        room_paxes = child_no+adults_no
        room_ages = all_ages[total_paxes:room_paxes+total_paxes]
        child_ages = room_ages[adults_no:]
        total_paxes+=room_paxes
        if child_no == 0:
            x+=f"""<RoomStayCandidate Quantity="1">
                    <GuestCounts><GuestCount Count="{adults_no}"/></GuestCounts>
                   </RoomStayCandidate>"""
        elif child_no != 0:
            ch = ""
            #total_fields = 2+(child_no)
            for c in range(child_no):
                ch += ""f'<GuestCount Age="{child_ages[c]}" Count="1" />'""
            x += f"""<RoomStayCandidate Quantity="1">
                <GuestCounts>
                <GuestCount Count="{adults_no}"/>
                {ch}
                </GuestCounts>
            </RoomStayCandidate>""" 
    try:
        result = await main([cleaned_df.at[row,'JPCode']], cleaned_df.at[row,'Travel Date'], cleaned_df.at[row,'End Date'], x, cleaned_df.at[row,'Nationality Id'])
        #result = await main([lost_df.at[row,'JPCode']], lost_df.at[row,'Travel Date'], lost_df.at[row,'End Date'], x, lost_df.at[row,'Nationality Id'])
        #print(result)
        result['BookingCode'] = cleaned_df.at[row,'BookingCode']
        result['Line Id'] = cleaned_df.at[row,'Line Id']
        availability_df = availability_df._append(result)
    except:
        pass
#print(lost_df.at[row,'BookingCode'],result.head(2))

  0%|          | 0/1636 [00:00<?, ?it/s]

In [16]:
availability_df.head()

,JPCode,HotelName,BoardType,RoomType,Rate,Refundable/Not,rateplan,BookingCode,Line Id
0,JP02875D,Panan Krabi Resort,Room Only,"Deluxe Family Pool View , 2 Queen Beds",646.26,NRF,vQANVTmkt548IDNLE1blqAnJZfqQcIVM1ZdUbVyQEa/Z35...,VYCB4K,1545464
1,JP02875D,Panan Krabi Resort,Room Only,"Deluxe Family Pool View , 2 Queen Beds",661.77,NRF,vQANVTmkt548IDNLE1blqAnJZfqQcIVM1ZdUbVyQEa+DHU...,VYCB4K,1545464
2,JP02875D,Panan Krabi Resort,ROOM ONLY,FAMILY ROOM DELUXE POOL VIEW,671.97,RF,qy2rKu1yosnIg0EUXXzE2mDFjwvj98GwFrZDyF/OUFkAlL...,VYCB4K,1545464
3,JP02875D,Panan Krabi Resort,Room only,Deluxe Family Pool View,672.88,NRF,8BQKKbHt3TpHE5iJgFcTCK4YWncpR3xIU3BqtNo5x7x7Dt...,VYCB4K,1545464
4,JP02875D,Panan Krabi Resort,ROOM ONLY,FAMILY ROOM DELUXE POOL VIEW,679.26,NRF,qy2rKu1yosnIg0EUXXzE2mDFjwvj98GwFrZDyF/OUFkfU6...,VYCB4K,1545464


In [17]:
#set(cleaned_df['BookingCode'].to_list()).difference(set(availability_df['BookingCode'].to_list()))

In [18]:
availability_df.shape

(248032, 9)

In [19]:
# availability_df = pd.DataFrame(columns=['Line Id','JPCode','BoardType','RoomType','Rate','Start_date','End_date','NonRefundable'])

# @retrying.retry(stop_max_attempt_number=3, wait_exponential_multiplier=1000, wait_exponential_max=10000)
# async def get_availability_xml(session,pbar,line_Id,start_date,end_date,Jp_code,Nationality,Number_of_paxes):
#     url="xxxxx/webservice/OTA_HotelAvail.asmx"
#     headers = {'content-type': 'text/xml;charset=UTF-8','SOAPAction':"http://www.opentravel.org/OTA/2003/05/OTA_HotelAvailService",
#                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:66.0) Gecko/20100101 Firefox/66.0",
#                "Accept-Encoding":"gzip,deflate","Connection": "keep-alive"}
#     body=f"""<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema">
#     <soap:Body>
#     <OTA_HotelAvailService xmlns="http://www.opentravel.org/OTA/2003/05">
#     <OTA_HotelAvailRQ PrimaryLangID="en">
#         <POS>
#             <Source AgentDutyCode="BookingZone_XML02">
#                 <RequestorID Type="1" MessagePassword="D+8mgtxJkAWT"/>
#             </Source>
#         </POS>
#         <AvailRequestSegments>
#             <AvailRequestSegment>
#                 <StayDateRange Start="{start_date}" End="{end_date}"/>
#                 <RoomStayCandidates>
#                     <RoomStayCandidate Quantity="1">
#                         <GuestCounts><GuestCount Count="{str(Number_of_paxes)}"/></GuestCounts>
#                     </RoomStayCandidate>
#                 </RoomStayCandidates>
#                 <HotelSearchCriteria>
#                     <Criterion><HotelRef HotelCode="{Jp_code}" />
#                         <TPA_Extensions>
#                             <ShowOnlyAvailable>1</ShowOnlyAvailable>
#                             <ShowInternalRoomInfo>1</ShowInternalRoomInfo>
#                             <ShowInternalProvInfo>1</ShowInternalProvInfo>
#                             <ShowCatalogueData>1</ShowCatalogueData>
#                             <PackageContracts>Package</PackageContracts>
#                             <ShowDailyAvailabilityBreakdown>0</ShowDailyAvailabilityBreakdown>
#                             <ShowBasicInfo>1</ShowBasicInfo>
#                             <PaxCountry>{Nationality}</PaxCountry>
#                             <SearchTimeOut>5000</SearchTimeOut>
#                         </TPA_Extensions>
#                     </Criterion>
#                 </HotelSearchCriteria>
#             </AvailRequestSegment>
#         </AvailRequestSegments>
#     </OTA_HotelAvailRQ>
#     </OTA_HotelAvailService>
#     </soap:Body>
#     </soap:Envelope>"""
#     try:
#         async with session.post(url, data=body, headers=headers) as response:
#             responseDict = xmltodict.parse(await response.text())
#             AllData = responseDict['soap:Envelope']['soap:Body']['OTA_HotelAvailServiceResponse']['OTA_HotelAvailRS']['RoomStays']['RoomStay']['RoomRates']['RoomRate']
#             for j in range(len(AllData)):
#                 row=[
#                     line_Id,
#                     Jp_code,
#                     AllData[j]['@RatePlanCategory'],
#                     AllData[j]['Rates']['Rate']['RateDescription']['Text'],
#                     AllData[j]['Rates']['Rate']['Total']['@AmountAfterTax'],
#                     start_date,
#                     end_date,
#                     AllData[j]['TPA_Extensions']['NonRefundable']['#text']
#                 ]
#                 availability_df.loc[len(availability_df)]=row
#     except:
#         pass
#     pbar.update(1)


# async def main():
#     for index in tqdm(range(0,cleaned_df.shape[0],200)):
#         if index + 200 > cleaned_df.shape[0]:
#             end_index = cleaned_df.shape[0]-1
#         else:
#             end_index = index + 200
#         sliced_df = cleaned_df.loc[index:end_index,:]
#         sliced_df = sliced_df.reset_index()
#         async with aiohttp.ClientSession() as session:
#             tasks = []
#             with tqdm(total=len(sliced_df), desc="Processing") as pbar:
#                 for i,v in sliced_df.iterrows():
#                     tasks.append(asyncio.ensure_future(get_availability_xml(session,pbar,sliced_df.iloc[i]['Line Id'],sliced_df.iloc[i]['Travel Date'],sliced_df.iloc[i]['End Date'],sliced_df.iloc[i]['JPCode'],sliced_df.iloc[i]['Nationality Id'],sliced_df.iloc[i]['No. Paxes'])))
#                 await asyncio.gather(*tasks)

# await main()

In [20]:
availability_df[availability_df['Refundable/Not']!='RF'].shape

(85559, 9)

In [21]:
availability_df = availability_df[availability_df['Refundable/Not']=='RF']

In [22]:
availability_df.shape

(162473, 9)

In [23]:
mapping = {'Bed and Breakfast': 'Breakfast','Room Only': 'Room Only', 'Half Board': 'Half Board', 'All inclusive': 'Inclusive',
 'Buffet Breakfast, Buffet Breakfast, Buffet Breakfast': 'Breakfast', 'Buffet Breakfast': 'Breakfast', 'Ultra All Inclusive': 'Inclusive',
 'Free breakfast': 'Breakfast', 'Breakfast & Dinner': 'Breakfast', 'Breakfast': 'Breakfast', 'Breakfast buffet': 'Breakfast',
 'Breakfast buffet, Breakfast buffet, Breakfast buffet, Breakfast buffet': 'Breakfast', 'Room only, Room only': 'Room Only',
 'Buffet Breakfast, Buffet Breakfast': 'Breakfast','Free breakfast, Free breakfast, Free breakfast, Free breakfast': 'Breakfast',
 'Full Board': 'Full Board', 'American Breakfast, American Breakfast': 'Breakfast',
 'Free breakfast, Free breakfast': 'Breakfast', 'Hard All inclusive': 'Inclusive', 'Bed and Breakfast, Bed and Breakfast, Bed and Breakfast': 'Breakfast',
 'American Breakfast': 'Breakfast', 'Breakfast, Breakfast': 'Breakfast', 'Bed and Breakfast, Bed and Breakfast': 'Breakfast',
 'Breakfast buffet, Breakfast buffet': 'Breakfast', 'Room only, Room only, Room only': 'Room Only', 'Premium All Inclusive': 'Inclusive',
 'Executive Floor Breakfast, Executive Floor Breakfast': 'Breakfast', 'Complimentary Breakfast': 'Breakfast', 'Soft All Inclusive ': 'Inclusive',
 'Breakfast for 1, Breakfast for 1': 'Breakfast','Continental breakfast': 'Breakfast', 'Full Breakfast': 'Breakfast',
 'Breakfast for 2, Breakfast for 2': 'Breakfast', 'Executive Floor Breakfast': 'Breakfast',
 'Breakfast, Breakfast, Breakfast, Breakfast': 'Breakfast','Breakfast for 2': 'Breakfast','Free breakfast, Free breakfast, Free breakfast': 'Breakfast',
 'All Inclusive, All Inclusive': 'Inclusive', 'All-inclusive': 'Inclusive', 'Room Only, Room Only, Room Only, Room Only': 'Room Only',
 'Continental breakfast, Continental breakfast': 'Breakfast','Full English Breakfast': 'Breakfast',
 'Bed and Breakfast, Bed and Breakfast, Bed and Breakfast, Bed and Breakfast, Bed and Breakfast': 'Breakfast','Breakfast, Breakfast, Breakfast': 'Breakfast',
 'Full Breakfast, Full Breakfast': 'Breakfast', 'Breakfast buffet, Breakfast buffet, Breakfast buffet': 'Breakfast',
 'SCOTTISH BREAKFAST': 'Breakfast', 'Ramadan Sohour ': 'Sohour', 'Half board, Half board': 'Half Board',
 'Free breakfast, Free breakfast, Free breakfast, Free breakfast, Free breakfast': 'Breakfast','Room only':'Room Only','Breakfast  Dinner':'Half Board',
 'BED AND BREAKFAST':'Breakfast','Half board':'Half Board','ROOM ONLY':'Room Only','Continental Breakfast':'Breakfast','Bed And breakfast': 'Breakfast',
 'All Inclusive': 'Inclusive','Alojamiento y Desayuno':'Breakfast','Full board':'Full Board','Room and Board':'Room and Board',
         'Free dinner':'Free dinner','Buffet dinner':'Buffet dinner','Half board (buffet dinner)':'Half Board','Sólo Alojamiento':'Room Only',
          'Soft All Inclusive':'Inclusive','Desayuno gratis':'Breakfast','Room Only/Sólo Alojamiento':'Room Only','All meals':'Inclusive',
          'Breakfast for 1':'Breakfast','Continental breakfast for 2':'Breakfast','All-inclusive (food/beverages/snacks)':'Inclusive',
          'Buffer Dinner':'Buffer Dinner','American breakfast':'Breakfast','Free lunch':'Free lunch','English breakfast':'Breakfast',
           'Bed and Breakfast/Alojamiento y Desayuno':'Breakfast','Bed and Breakfast for 2 guests':'Breakfast',
         'All Inclusive for 2 guests':'Inclusive', 'Breakfast for two guests':'Breakfast','Bed & Breakfast':'Breakfast','Lunch *':'Lunch',
         'Dinner *':'Dinner','HALF BOARD':'Half Board','FULL BOARD':'Full Board','Bed&Breakfast':'Breakfast',
          'Bed and Breakfast for 4 guests':'Breakfast','CONTINENTAL BREAKFAST':'Breakfast','BUFFET BREAKFAST':'Breakfast',
          'DINNER INCLUDED':'Dinner','Half Board Plus':'Half Board','ALL INCLUSIVE':'Inclusive','Bed and Breakfast for 1 guest':'Breakfast',
          'Bed and Breakfast for 6 guests':'Breakfast','All Inclusive for 3 guests':'Inclusive'}

In [24]:
def mapBoard(x):
    
    if 'breakfast and dinner' in x.lower():
        return 'Breakfast and Dinner'
    elif 'breakfast' in x.lower() or 'alojamiento y desayuno' in x.lower():
        return 'Breakfast'
    elif 'inclusive' in x.lower():
        return 'Inclusive'
    elif 'room only' in x.lower():
        return 'Room Only'
    elif 'half' in x.lower():
        return 'Half Board'
    elif 'full' in x.lower():
        return 'Full Board'
    elif 'lunch' in x.lower():
        return 'Lunch'
    elif 'dinner' in x.lower():
        return 'Dinner'
    elif "sólo alojamiento" in x.lower():
        return 'Room Only'
    elif "room and board" in x.lower():
        return 'Room and Board'

In [25]:
cleaned_df['Mapped_Board']=cleaned_df['Board Type'].apply(lambda x : mapBoard(x))

In [26]:
cleaned_df[cleaned_df['Mapped_Board'].isna()]['Board Type'].unique()

array(['nomeal'], dtype=object)

In [27]:
availability_df['Mapped_Board']=availability_df['BoardType'].apply(lambda x : mapBoard(x))

In [28]:
availability_df[availability_df['Mapped_Board'].isna()]['BoardType'].unique()

array(['SELF CATERING'], dtype=object)

In [29]:
lines_ids = cleaned_df['Line Id'].to_list()

In [30]:
cleaned_df = cleaned_df.reset_index()

In [31]:
matching_df = pd.DataFrame(columns=['Line Id','Matching_room','Matching_cost','Matching_board','%Matching'])

for id in tqdm(lines_ids):
    try:
        ord_df = cleaned_df[cleaned_df['Line Id']==id]
        org_board = ord_df['Mapped_Board'].values[0]
        av_df = availability_df[(availability_df['Line Id']==id) & (availability_df['Mapped_Board']==org_board)]
        av_df = av_df.reset_index()
        org_room = ord_df['Room Name'].values[0]
        best_score = 0
        matching_room = ''
        matching_cost = ''
        matching_board = ''
        org_room = org_room.replace(" DBL "," Double ").replace(" SGL ",' single ').replace(" TPL "," Triple ").replace('  ',' ')
        for i,v in av_df.iterrows():
            av_room = av_df.iloc[i]['RoomType']
            av_room = av_room.replace(" DBL "," Double ").replace(" SGL ",' single ').replace(" TPL "," Triple ").replace('  ',' ')
            score = fuzz.QRatio(org_room.lower().strip(), av_room.lower().strip())
            if score >= 90:
                row=[id,av_df.iloc[i]['RoomType'],av_df.iloc[i]['Rate'],av_df.iloc[i]['BoardType'],score]
                matching_df.loc[len(matching_df)]=row
    except:
        row=[id,'Not Available','Not Available','Not Available',0]
        matching_df.loc[len(matching_df)]=row

  0%|          | 0/1636 [00:00<?, ?it/s]

In [32]:
cleaned_df = cleaned_df.merge(matching_df,on='Line Id',how='left')

In [33]:
# for id in tqdm(lines_ids):
#     try:
#         ord_df = cleaned_df[cleaned_df['Line Id']==id]
#         org_board = ord_df['Mapped_Board'].values[0]
#         av_df = availability_df[(availability_df['Line Id']==id) & (availability_df['Mapped_Board']==org_board)]
#         av_df = av_df.reset_index()
#         org_room = ord_df['Room Name'].values[0]
#         best_score = 0
#         matching_room = ''
#         matching_cost = ''
#         matching_board = ''
#         org_room = org_room.replace(" DBL "," Double ").replace(" SGL ",' single ').replace(" TPL "," Triple ").replace('  ',' ')
#         for i,v in av_df.iterrows():
#             av_room = av_df.iloc[i]['RoomType']
#             av_room = av_room.replace(" DBL "," Double ").replace(" SGL ",' single ').replace(" TPL "," Triple ").replace('  ',' ')
#             score = fuzz.QRatio(org_room.lower().strip(), av_room.lower().strip())
#             if score > best_score:
#                 best_score = score
#                 matching_room = av_df.iloc[i]['RoomType']
#                 matching_cost = av_df.iloc[i]['Rate']
#                 matching_board = av_df.iloc[i]['BoardType']
#         cleaned_df.loc[cleaned_df['Line Id']==id,'Matching_room'] = matching_room
#         cleaned_df.loc[cleaned_df['Line Id']==id,'Matching_cost'] = matching_cost
#         cleaned_df.loc[cleaned_df['Line Id']==id,'Matching_board'] = matching_board
#         cleaned_df.loc[cleaned_df['Line Id']==id,'%Matching'] = best_score
#     except:
#         cleaned_df.loc[cleaned_df['Line Id']==id,'Matching_room'] = 'Not Available'
#         cleaned_df.loc[cleaned_df['Line Id']==id,'Matching_cost'] = 'Not Available'
#         cleaned_df.loc[cleaned_df['Line Id']==id,'Matching_board'] = 'Not Available'
#         cleaned_df.loc[cleaned_df['Line Id']==id,'%Matching'] = best_score
    

In [34]:
cleaned_df = cleaned_df[(cleaned_df['Matching_cost']!='') & (cleaned_df['Matching_cost']!='Not Available')]

In [35]:
cleaned_df['Net Cost(USD)'] = cleaned_df['Net Cost(USD)'].astype(float)
cleaned_df['Matching_cost'] = cleaned_df['Matching_cost'].astype(float)

In [36]:
cleaned_df['diff']=cleaned_df['Net Cost(USD)']-cleaned_df['Matching_cost']

In [37]:
final_result = cleaned_df[(cleaned_df['diff']>=20) & (cleaned_df['Days To Cancel']>2) & (cleaned_df['%Matching']>=95)]

In [38]:
final_result = final_result.sort_values(by='diff',ascending=False)

In [39]:
final_result['Net Cost(USD)'] = round(final_result['Net Cost(USD)'],2)
final_result['Matching_cost'] = round(final_result['Matching_cost'],2)
final_result['diff'] = round(final_result['diff'],2)

In [40]:
final_result = final_result.sort_values(by='diff',ascending=False)

In [41]:
final_result = final_result.drop('level_0', axis=1).reset_index(drop=True)
final_result=final_result.reset_index()

In [42]:
booking_codes_list = []
for i,v in final_result.iterrows():
    if final_result.iloc[i]['BookingCode'] not in booking_codes_list:
        booking_codes_list.append(final_result.iloc[i]['BookingCode'])

In [43]:
final_result['%Matching'] = final_result['%Matching'].apply(lambda x: '{:.2f}%'.format(x))

In [44]:
final_result.shape

(61, 34)

In [45]:
#final_result = final_result[final_result['Agency']!="Contracting Test (ALL NATIONALITY)"]

In [46]:
children=[]
z=1
v=str(z)
length=len(booking_codes_list)

for booking_code in tqdm(booking_codes_list):
    
    df_booking_code=final_result[final_result["BookingCode"]==booking_code]
    df_header_table_1=df_booking_code[['BookingCode','Agency',"JPCode",'HotelName','Travel Date','End Date']].drop_duplicates()
    df_header_table_2=df_booking_code[['Nationality','No. Paxes','Supplier Name','Days To Cancel','Internal Remarks','Booking Remarks']].drop_duplicates()
    df_booking_table=df_booking_code[["Board Type","Room Name" ,"Net Cost(USD)"]].drop_duplicates()
    df_lower_alternatives=df_booking_code[["Matching_board","Matching_room","Matching_cost","diff",'%Matching']].drop_duplicates()

    if df_booking_code.empty==False:
        Div_Component=html.Div([
        html.P(f"({z} / {len(booking_codes_list)}) Booking Code {df_booking_code['BookingCode'].values[0]}",style={"text-align": "center","font-family":'verdana',"font-size":"20px",'white': 'black','fontWeight': 'bold'}),
            html.P(f"Booking Details",style={"text-align": "center","font-family":'verdana',"font-size":"20px",'white': 'black','fontWeight': 'bold'})
            ,dbc.Card(
            dbc.CardBody(dash_table.DataTable(
            id=f'{booking_code}_{v}_6',
            columns=[{"name": i, "id": i} 
                     for i in df_header_table_1.columns],
            sort_action='native',
                data=df_header_table_1.to_dict('records')
                ,editable=True,       
        style_cell={'whiteSpace': 'normal','textAlign': 'center'},style_header={
            'backgroundColor': 'rgb(210, 210, 210)',
            'color': 'black',
            'fontWeight': 'bold',
            'border': '1px solid black','height': 'auto',
        'minWidth': '180px', 'width': '300px', 'maxWidth': '180px',
        'overflow': 'hidden',
        'textOverflow': 'ellipsis'
        },style_data={
            'backgroundColor': 'rgb(255, 255, 255)',
            'color': 'black',
            'border': '1px solid black'
        },cell_selectable=False))),html.P(f"_____",style={"text-align": "center","font-family":'verdana',"font-size":"20px",'white': 'black','fontWeight': 'bold'})
            ,
            dbc.Card(
            dbc.CardBody(dash_table.DataTable(
            id=f'{booking_code}_{v}_10',
            columns=[{"name": i, "id": i} 
                     for i in df_header_table_2.columns],
            sort_action='native',
                data=df_header_table_2.to_dict('records')
                ,editable=True,       
        style_cell={'whiteSpace': 'normal','textAlign': 'center'},style_header={
            'backgroundColor': 'rgb(210, 210, 210)',
            'color': 'black',
            'fontWeight': 'bold',
            'border': '1px solid black','height': 'auto',
        # all three widths are needed
        'minWidth': '180px', 'width': '300px', 'maxWidth': '180px',
        'overflow': 'hidden',
        'textOverflow': 'ellipsis'
        },style_data={
            'backgroundColor': 'rgb(255, 255, 255)',
            'color': 'black',
            'border': '1px solid black'
        },cell_selectable=False))),
            html.P(f"Original Booking Rate",style={"text-align": "center","font-family":'verdana',"font-size":"20px",'white': 'black','fontWeight': 'bold'}),
            dbc.Card(
            dbc.CardBody(dash_table.DataTable(
            id=f'{booking_code}_{v}_1',
            columns=[{"name": i, "id": i} 
                     for i in df_booking_table.columns],
            data=df_booking_table.to_dict('records'),
            style_cell={'whiteSpace': 'normal','textAlign': 'center'},style_header={
            'backgroundColor': 'rgb(210, 210, 210)',
            'color': 'black',
            'fontWeight': 'bold',
            'border': '1px solid black',
                'height': 'auto',
        # all three widths are needed
        'minWidth': '180px', 'width': '180px', 'maxWidth': '180px',
        'overflow': 'hidden',
        'textOverflow': 'ellipsis'
        },cell_selectable=False,style_data={
            'backgroundColor': 'rgb(255, 255, 255)',
            'color': 'black',
            'border': '1px solid black'
        },))),
            html.P(f"Lower Rates Alternatives",style={"text-align": "center","font-family":'verdana',"font-size":"20px",'white': 'black','fontWeight': 'bold'}),
            html.P(f"Please be aware that these are the most similar options not necessarily the same",style={"text-align": "center","font-family":'verdana',"font-size":"15px",'color': 'blue'}),
            html.P(f"Make sure to check the room name for exact match",style={"text-align": "center","font-family":'verdana',"font-size":"15px",'color': 'blue'}),
            dbc.Card(
            dbc.CardBody(dash_table.DataTable(
            id=f'{booking_code}_{v}_2',
            columns=[{"name": i, "id": i} 
                     for i in df_lower_alternatives.columns],
            sort_action='native',
                data=df_lower_alternatives.to_dict('records')
                ,editable=True
        ,
                
        style_cell={'whiteSpace': 'normal','textAlign': 'center'},style_header={
            'backgroundColor': 'rgb(210, 210, 210)',
            'color': 'black',
            'fontWeight': 'bold',
            'border': '1px solid black','height': 'auto',
        # all three widths are needed
        'minWidth': '180px', 'width': '300px', 'maxWidth': '180px',
        'overflow': 'hidden',
        'textOverflow': 'ellipsis'
        },style_data={
            'backgroundColor': 'rgb(255, 255, 255)',
            'color': 'black',
            'border': '1px solid black'
        },cell_selectable=False))),
        html.Br()],style={'margin-left':10,'marginBottom': 10, 'marginTop': 10,'border':'5px solid black'})
        children.append(Div_Component)
        z+=1
    else:
        pass
    
    
#Get Run Time & Update Time

now = datetime.now()
current_time = now.strftime("'%Y-%m-%d %I:%M %p'")
current_time
#Display Update Time
Div_Component=html.P(f" Last Update Time {current_time}",style={"text-align": "center","font-family":'verdana',"font-size":"40px",'white': 'black','fontWeight': 'bold'})
children.insert(0,Div_Component)
Div_Component=html.P(f" Disclaimer : This is an Internal Report",style={"text-align": "center","font-family":'verdana',"font-size":"25px",'color': 'red'})
children.insert(1,Div_Component)

app = JupyterDash(external_stylesheets=[dbc.themes.SANDSTONE])
app.layout = html.Div(children=children)

@app.callback(
    Output(f'table_out-{booking_code}_{v}_1', 'children'), 
    Input(f'{booking_code}_{v}_1', 'active_cell_1'),
    Input(f'{booking_code}_{v}_2', 'active_cell_2'))

def update_graphs(active_cell_1,active_cell_2):
    if active_cell_1:
        return "Click the table"
        
print('done')
if __name__ == '__main__':
    # Run the Dash application
    app.run_server(debug=False,port=8051)


  0%|          | 0/46 [00:00<?, ?it/s]

done


Dash app running on http://127.0.0.1:8051/


In [47]:
#  http://127.0.0.1:8051/